In [ ]:
!pip install rdkit -q
!pip install chembl-webresource-client -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

# RDKit core
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, rdMolDescriptors
from rdkit.Chem.Draw import MolToImage
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator

# Fingerprints
from rdkit.Chem import AllChem
from rdkit.Chem import MACCSkeys


plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        120,
})

print(f'RDKit version: {Chem.rdBase.rdkitVersion}')
print('All libraries imported.')

In [ ]:
# A diverse set of approved drugs covering a range of sizes and properties
DRUGS = [
    ('Aspirin',      'CC(=O)Oc1ccccc1C(=O)O'),
    ('Caffeine',     'Cn1cnc2c1c(=O)n(C)c(=O)n2C'),
    ('Ibuprofen',    'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
    ('Paracetamol',  'CC(=O)Nc1ccc(O)cc1'),
    ('Ciprofloxacin','O=C(O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O'),
    ('Sildenafil',   'CCCC1=NN(C)C(=C1C(=O)NCC2=CC(=CC=C2OCC)S(=O)(=O)N3CCN(CC3)C)C4=CC=CC=C4'),
    ('Erlotinib',    'C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1'),
    ('Imatinib',     'Cc1ccc(NC(=O)c2ccc(CN3CCN(CC3)C)cc2)cc1Nc1nccc(-c2cccnc2)n1'),
]

# Convert all to RDKit molecule objects
mols = [(name, Chem.MolFromSmiles(smi)) for name, smi in DRUGS]

# Validate -- MolFromSmiles returns None for invalid SMILES
for name, mol in mols:
    status = 'OK' if mol is not None else 'INVALID SMILES'
    print(f'  {name:20s}: {status}')

In [ ]:
# Walk through a few SMILES examples from simple to complex
examples = [
    ('Ethanol',   'CCO',                    'Two carbons, one oxygen'),
    ('Benzene',   'c1ccccc1',               'Six aromatic carbons in a ring'),
    ('Aspirin',   'CC(=O)Oc1ccccc1C(=O)O', 'Benzene ring + two functional groups'),
    ('Invalid',   'CC(=O)Oc1ccccc1C(=O',   'Deliberately broken SMILES'),
]

print(f'{"Name":12s}  {"SMILES":35s}  {"Valid":6s}  {"Atoms":6s}  {"Note"}')
print('-' * 80)
for name, smi, note in examples:
    mol    = Chem.MolFromSmiles(smi)
    valid  = mol is not None
    n_atoms = mol.GetNumAtoms() if valid else '-'
    print(f'{name:12s}  {smi:35s}  {str(valid):6s}  {str(n_atoms):6s}  {note}')
     

In [ ]:
# Canonicalisation -- RDKit always produces the same SMILES for the same molecule
# regardless of which atom you start from
aspirin_variants = [
    'CC(=O)Oc1ccccc1C(=O)O',   # standard
    'OC(=O)c1ccccc1OC(C)=O',   # different starting atom
    'O=C(O)c1ccccc1OC(=O)C',   # yet another traversal
]

print('Canonicalising aspirin SMILES variants:')
for smi in aspirin_variants:
    mol       = Chem.MolFromSmiles(smi)
    canonical = Chem.MolToSmiles(mol)
    print(f'  Input:     {smi}')
    print(f'  Canonical: {canonical}')
    print()
     

In [ ]:
# Draw all eight reference drugs in a grid
mol_list   = [mol for _, mol in mols]
name_list  = [name for name, _ in mols]

img = Draw.MolsToGridImage(
    mol_list,
    molsPerRow=4,
    subImgSize=(320, 220),
    legends=name_list,
)
display(img)

In [ ]:
def morgan_fp(mol, radius=2, n_bits=1024):
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fp  = mfpgen.GetFingerprint(mol)
    arr = np.zeros(n_bits, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


# Compute for all drugs
fp_matrix = np.vstack([morgan_fp(mol) for _, mol in mols])

print(f'Fingerprint matrix shape: {fp_matrix.shape}')
print(f'  Rows = {fp_matrix.shape[0]} molecules')
print(f'  Cols = {fp_matrix.shape[1]} bits')
print(f'\nBits set per molecule:')
for (name, _), fp in zip(mols, fp_matrix):
    print(f'  {name:20s}: {fp.sum():3d} / {len(fp)} bits set  ({fp.sum()/len(fp)*100:.1f}%)')

In [ ]:
# Visualise the fingerprint matrix as a heatmap
# Show only the first 200 bits for clarity -- the pattern is the same across all 1024
N_BITS_SHOW = 200

fig, ax = plt.subplots(figsize=(13, 4))

im = ax.imshow(
    fp_matrix[:, :N_BITS_SHOW],
    aspect='auto',
    cmap='Blues',
    interpolation='nearest',
)

ax.set_yticks(range(len(mols)))
ax.set_yticklabels([name for name, _ in mols], fontsize=10)
ax.set_xlabel(f'Bit position (showing first {N_BITS_SHOW} of 1024)', fontsize=11)
ax.set_title(
    'Morgan Fingerprints (ECFP4, radius=2)\n'
    'Blue = bit set (substructure present)   White = bit not set',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

print('Each row is one molecule. Each column is one bit position.')
print('Where two molecules share a blue column they share that substructure.')

In [ ]:
# Effect of radius -- how the fingerprint changes as we capture larger neighbourhoods
aspirin_mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
colors = ['#0D9488', '#1B2A4A', '#E57373']

for ax, radius, color in zip(axes, [0, 1, 2], colors):
    fp  = morgan_fp(aspirin_mol, radius=radius)
    ax.bar(range(100), fp[:100], color=color, width=1.0, alpha=0.8)
    ax.set(title=f'Radius {radius}  ({fp.sum()} bits set)',
           xlabel='Bit position (first 100)', ylabel='Bit value')
    ax.set_yticks([0, 1])

fig.suptitle('Effect of Radius on Morgan Fingerprint (Aspirin)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
     

In [ ]:

def maccs_fp(mol):
    """Compute MACCS keys fingerprint as a numpy array (166 bits)."""
    fp  = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros(167, dtype=np.uint8)   # RDKit returns 167 bits, bit 0 unused
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr[1:]   # return bits 1-166


maccs_matrix = np.vstack([maccs_fp(mol) for _, mol in mols])

fig, ax = plt.subplots(figsize=(13, 4))
ax.imshow(maccs_matrix, aspect='auto', cmap='Purples', interpolation='nearest')
ax.set_yticks(range(len(mols)))
ax.set_yticklabels([name for name, _ in mols], fontsize=10)
ax.set_xlabel('MACCS key bit position (1-166)', fontsize=11)
ax.set_title(
    'MACCS Keys Fingerprints\n'
    'Purple = key present   White = key absent   (each bit has a defined chemical meaning)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
# Compare Morgan vs MACCS: which fingerprint captures more similarity
# between structurally related compounds?
# Erlotinib and Imatinib are both kinase inhibitors -- do the fingerprints reflect this?

from rdkit import DataStructs as DS

erlotinib = Chem.MolFromSmiles('C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1')
imatinib  = Chem.MolFromSmiles('Cc1ccc(NC(=O)c2ccc(CN3CCN(CC3)C)cc2)cc1Nc1nccc(-c2cccnc2)n1')
aspirin   = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')

def tanimoto_morgan(m1, m2, radius=2, n_bits=1024):
    from rdkit.Chem import rdFingerprintGenerator
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fp1 = mfpgen.GetFingerprint(m1)
    fp2 = mfpgen.GetFingerprint(m2)
    return DS.TanimotoSimilarity(fp1, fp2)

def tanimoto_maccs(m1, m2):
    fp1 = MACCSkeys.GenMACCSKeys(m1)
    fp2 = MACCSkeys.GenMACCSKeys(m2)
    return DS.TanimotoSimilarity(fp1, fp2)

pairs = [
    ('Erlotinib vs Imatinib (both kinase inhibitors)', erlotinib, imatinib),
    ('Erlotinib vs Aspirin (unrelated)',               erlotinib, aspirin),
    ('Imatinib vs Aspirin (unrelated)',                imatinib,  aspirin),
]

print(f'{"Pair":45s}  {"Morgan Tanimoto":16s}  {"MACCS Tanimoto"}')
print('-' * 80)
for label, m1, m2 in pairs:
    t_morgan = tanimoto_morgan(m1, m2)
    t_maccs  = tanimoto_maccs(m1, m2)
    print(f'{label:45s}  {t_morgan:.3f}            {t_maccs:.3f}')

In [ ]:

def compute_descriptors(mol):
    """Compute a standard set of physicochemical descriptors for one molecule."""
    return {
        'MW':   round(Descriptors.MolWt(mol), 2),
        'LogP': round(Descriptors.MolLogP(mol), 2),
        'HBD':  rdMolDescriptors.CalcNumHBD(mol),
        'HBA':  rdMolDescriptors.CalcNumHBA(mol),
        'RotB': rdMolDescriptors.CalcNumRotatableBonds(mol),
        'TPSA': round(rdMolDescriptors.CalcTPSA(mol), 2),
        'Rings':rdMolDescriptors.CalcNumRings(mol),
        'ArRings': rdMolDescriptors.CalcNumAromaticRings(mol),
    }


desc_rows = []
for name, mol in mols:
    row = {'Name': name}
    row.update(compute_descriptors(mol))
    desc_rows.append(row)

desc_df = pd.DataFrame(desc_rows).set_index('Name')
desc_df

In [ ]:
# Visualise descriptors as a parallel coordinates-style bar chart
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes      = axes.flatten()
cols      = ['MW', 'LogP', 'HBD', 'HBA', 'RotB', 'TPSA', 'Rings', 'ArRings']
colors    = plt.cm.tab10(np.linspace(0, 1, len(mols)))

for ax, col in zip(axes, cols):
    bars = ax.bar(range(len(mols)), desc_df[col], color=colors, alpha=0.85)
    ax.set_xticks(range(len(mols)))
    ax.set_xticklabels([n for n, _ in mols], rotation=35, ha='right', fontsize=8)
    ax.set_title(col, fontweight='bold', fontsize=11)
    ax.set_ylabel(col, fontsize=9)

fig.suptitle('Physicochemical Descriptors Across Eight Drugs',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
     

In [ ]:
def lipinski_check(row):
    """Return individual rule pass/fail and overall result."""
    rules = {
        'MW <= 500':   row['MW']   <= 500,
        'LogP <= 5':   row['LogP'] <= 5,
        'HBD <= 5':    row['HBD']  <= 5,
        'HBA <= 10':   row['HBA']  <= 10,
    }
    violations = sum(1 for v in rules.values() if not v)
    rules['Violations'] = violations
    rules['Passes Ro5'] = violations <= 1
    return rules


ro5_rows = []
for name, row in desc_df.iterrows():
    result = lipinski_check(row)
    result['Name'] = name
    ro5_rows.append(result)

ro5_df = pd.DataFrame(ro5_rows).set_index('Name')
ro5_df

In [ ]:
# Visualise Ro5 pass/fail with a traffic-light style plot
rule_cols  = ['MW <= 500', 'LogP <= 5', 'HBD <= 5', 'HBA <= 10']
drug_names = ro5_df.index.tolist()

fig, ax = plt.subplots(figsize=(9, 5))

for r_idx, rule in enumerate(rule_cols):
    for d_idx, drug in enumerate(drug_names):
        passed = ro5_df.loc[drug, rule]
        color  = '#0D9488' if passed else '#E57373'
        ax.scatter(r_idx, d_idx, c=color, s=300, zorder=3)
        symbol = 'pass' if passed else 'FAIL'
        ax.text(r_idx, d_idx, symbol,
                ha='center', va='center', fontsize=7,
                color='white', fontweight='bold')

ax.set_xticks(range(len(rule_cols)))
ax.set_xticklabels(rule_cols, fontsize=11)
ax.set_yticks(range(len(drug_names)))
ax.set_yticklabels(drug_names, fontsize=10)
ax.set_xlim(-0.5, len(rule_cols) - 0.5)
ax.set_ylim(-0.5, len(drug_names) - 0.5)
ax.grid(True, alpha=0.2)
ax.set_title('Lipinski Rule of Five -- Pass / Fail per Drug',
             fontsize=12, fontweight='bold')

pass_patch = mpatches.Patch(color='#0D9488', label='Pass')
fail_patch = mpatches.Patch(color='#E57373', label='Fail')
ax.legend(handles=[pass_patch, fail_patch], loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

print('\nSummary:')
for drug in drug_names:
    v   = ro5_df.loc[drug, 'Violations']
    res = ro5_df.loc[drug, 'Passes Ro5']
    print(f'  {drug:20s}: {v} violation(s)  --> {"PASS" if res else "FAIL"}')

In [ ]:
from chembl_webresource_client.new_client import new_client

print('Querying ChEMBL for EGFR IC50 data...')
activity = new_client.activity

egfr_raw = activity.filter(
    target_chembl_id='CHEMBL203',
    standard_type='IC50',
    standard_units='nM',
).only([
    'molecule_chembl_id',
    'canonical_smiles',
    'standard_value',
    'standard_units',
])[:500]

egfr_df = pd.DataFrame(list(egfr_raw))
egfr_df['standard_value'] = pd.to_numeric(egfr_df['standard_value'], errors='coerce')
egfr_df = egfr_df.dropna(subset=['canonical_smiles', 'standard_value'])
egfr_df = egfr_df[egfr_df['standard_value'] > 0]

print(f'Retrieved {len(egfr_df)} EGFR IC50 records')
egfr_df.head()

In [ ]:
# Convert SMILES to RDKit molecules, drop any that fail
egfr_df['mol'] = egfr_df['canonical_smiles'].apply(Chem.MolFromSmiles)
egfr_df        = egfr_df[egfr_df['mol'].notna()].copy()
print(f'{len(egfr_df)} compounds with valid structures')

In [ ]:
# Compute descriptors for all EGFR compounds
desc_list = [compute_descriptors(mol) for mol in egfr_df['mol']]
egfr_desc = pd.DataFrame(desc_list, index=egfr_df.index)

# Compute Morgan fingerprints and add as separate columns
fp_arrays  = np.vstack([morgan_fp(mol) for mol in egfr_df['mol']])
fp_cols    = [f'fp_{i}' for i in range(fp_arrays.shape[1])]
fp_df      = pd.DataFrame(fp_arrays, columns=fp_cols, index=egfr_df.index)

# Combine into one table -- this is the QSAR-ready dataframe
qsar_df = pd.concat([
    egfr_df[['molecule_chembl_id', 'standard_value']].rename(
        columns={'standard_value': 'IC50_nM'}),
    egfr_desc,
    fp_df,
], axis=1)

print(f'QSAR-ready dataframe shape: {qsar_df.shape}')
print(f'  {len(egfr_desc.columns)} descriptor columns')
print(f'  {len(fp_cols)} fingerprint bit columns')
print(f'  1 activity label column (IC50_nM)')
qsar_df[['molecule_chembl_id', 'IC50_nM', 'MW', 'LogP', 'HBD', 'HBA', 'TPSA']].head(8)

In [ ]:
# IC50 distribution -- log scale is appropriate for potency data
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(qsar_df['IC50_nM'], bins=40, color='#1B2A4A', alpha=0.8, edgecolor='white')
axes[0].set(xlabel='IC50 (nM)', ylabel='Count', title='IC50 distribution (linear scale)')

log_ic50 = np.log10(qsar_df['IC50_nM'])
axes[1].hist(log_ic50, bins=40, color='#0D9488', alpha=0.8, edgecolor='white')
axes[1].set(xlabel='log10(IC50 / nM)', ylabel='Count', title='IC50 distribution (log scale)')
axes[1].axvline(log_ic50.median(), color='#E57373', lw=2, ls='--',
                label=f'Median: {10**log_ic50.median():.0f} nM')
axes[1].legend()

fig.suptitle('EGFR Inhibitor IC50 Distribution (ChEMBL)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
